In [2]:
import os, sys
sys.path.append(os.getenv("FALCON_PATH"))
print("sys.path:", sys.path)

sys.path: ['/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python314.zip', '/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14', '/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload', '', '/Users/ajayumashankar25gmail.com/Desktop/CAPSTONE/implementation/venv/lib/python3.14/site-packages', '/opt/homebrew/opt/python-tk@3.14/libexec', '/Users/ajayumashankar25gmail.com/Desktop/CAPSTONE/implementation/falcon_py']


In [3]:
import time
import json
from falcon import Falcon

In [4]:
def create_command(command_id, payload):
    return {
        "command_id": command_id,
        "timestamp": int(time.time()),
        "payload": payload
    }

In [5]:
def serialize_command(command: dict) -> bytes:
    return json.dumps(
        command,
        sort_keys=True,
        separators=(",", ":")
    ).encode()

In [6]:
model = Falcon(512)

secret_key, public_key = model.keygen()
print("Secret Key length:", len(secret_key))
print("Public Key length:", len(public_key))

Secret Key length: 6
Public Key length: 896


In [7]:
command = create_command("MOVE_FORWARD", {"speed": 5})

serialized = serialize_command(command)
signature = model.sign(secret_key, serialized)

print("Command signed")
print("Signature length:", len(signature))

Command signed
Signature length: 666


In [8]:
def verify_command(command, signature, public_key):
    MAX_AGE_SECONDS = 10
    serialized = serialize_command(command)

    # Cryptographic verification
    output = model.verify(public_key, serialized, signature)
    
    # Replay protection
    age = int(time.time()) - command["timestamp"]
    if age > MAX_AGE_SECONDS:
        raise Exception("Command expired")

    return output

In [9]:
try:
    output = verify_command(command, signature, public_key)
    if output:
        print("VALID COMMAND → ACCEPTED")
    else:
        print("REJECTED: Invalid signature")
except Exception as e:
    print("REJECTED:", e)

VALID COMMAND → ACCEPTED


In [10]:
time.sleep(11)

try:
    verify_command(command, signature, public_key)
    print("UNEXPECTED ACCEPT")
except Exception as e:
    print("OLD COMMAND → REJECTED (EXPECTED)")

OLD COMMAND → REJECTED (EXPECTED)


In [11]:
command2 = create_command("TURN_LEFT", {"angle": 90})
signature2 = model.sign(secret_key, serialize_command(command2))
try:
    output = verify_command(command2, signature2, public_key)
    if output:
        print("VALID COMMAND → ACCEPTED")
    else:
        print("REJECTED: Invalid signature")
except Exception as e:
    print("REJECTED:", e)

VALID COMMAND → ACCEPTED


In [12]:
time.sleep(12)
try:
    output = verify_command(command2, signature2, public_key)
    if output:
        print("VALID COMMAND → ACCEPTED")
    else:
        print("REJECTED: Invalid signature")
except Exception as e:
    print("REJECTED:", e)

REJECTED: Command expired
